# Hypothesis Testing Notebook

## Objective
Conduct statistical hypothesis testing to validate insights derived from exploratory data analysis and draw evidence-based conclusions about food price inflation trends.

## Learning Outcomes Addressed
- **LO1**: Apply core principles of statistics and probability
- **LO2**: Demonstrate practical data manipulation skills with Python
- **LO3**: Analyse real-world problems using data analytics
- **LO6**: Address ethical considerations in data analytics
- **LO8**: Communicate complex insights effectively

## Hackathon Outcomes Addressed
- **O6.1**: Develop clear and testable hypotheses
- **O6.2**: Conduct appropriate statistical tests
- **O6.3**: Generate actionable insights

---
## 1. Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Statistical testing
from scipy import stats
from scipy.stats import (
    shapiro, 
    levene, 
    ttest_ind, 
    mannwhitneyu,
    f_oneway,
    kruskal,
    pearsonr,
    spearmanr,
    chi2_contingency
)
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

---
## 2. Load Data

In [ ]:
# Load cleaned dataset
df = pd.read_csv('../data/cleaned/food_prices_cleaned.csv', parse_dates=['date'])

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nCountries: {df['country'].nunique()}")
print(f"Date Range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")

In [ ]:
# Filter data with available inflation values for hypothesis testing
df_inflation = df[df['inflation'].notna()].copy()
print(f"Records with inflation data: {len(df_inflation)}")

---
## 3. Hypothesis Formulation

Based on exploratory data analysis, we formulate the following hypotheses:

### Hypothesis 1: Regional Inflation Differences
**H₀ (Null)**: There is no significant difference in average inflation rates between different countries.  
**H₁ (Alternative)**: There is a significant difference in average inflation rates between different countries.

### Hypothesis 2: Price Volatility and Inflation Correlation
**H₀ (Null)**: There is no significant correlation between price volatility (price_range) and inflation.  
**H₁ (Alternative)**: There is a significant positive correlation between price volatility and inflation.

### Hypothesis 3: Seasonal Patterns in Food Prices
**H₀ (Null)**: There is no significant difference in average food prices across different months.  
**H₁ (Alternative)**: There is a significant difference in average food prices across different months.

### Hypothesis 4: Price Trends Over Time
**H₀ (Null)**: Food prices have not significantly increased over the study period.  
**H₁ (Alternative)**: Food prices have significantly increased over the study period.

---
## 4. Statistical Test Selection Framework

| Test Scenario | Assumptions | Test Used |
|--------------|-------------|----------|
| Compare 2 groups (normal) | Normality + Equal variance | Independent t-test |
| Compare 2 groups (non-normal) | Non-parametric | Mann-Whitney U |
| Compare >2 groups (normal) | Normality + Equal variance | ANOVA |
| Compare >2 groups (non-normal) | Non-parametric | Kruskal-Wallis |
| Correlation (normal) | Normality | Pearson correlation |
| Correlation (non-normal) | Non-parametric | Spearman correlation |

**Significance Level**: α = 0.05

---
## 5. Assumption Testing

### 5.1 Normality Tests

In [ ]:
def test_normality(data, variable_name, alpha=0.05):
    """
    Test normality using Shapiro-Wilk test (for samples <= 5000)
    and D'Agostino-Pearson test for larger samples.
    """
    # Sample if data is too large for Shapiro-Wilk
    if len(data) > 5000:
        sample_data = data.sample(n=5000, random_state=42)
    else:
        sample_data = data
    
    # Shapiro-Wilk test
    stat, p_value = shapiro(sample_data.dropna())
    
    is_normal = p_value > alpha
    
    print(f"\n{variable_name}:")
    print(f"  Shapiro-Wilk statistic: {stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    print(f"  Conclusion: {'Normal distribution' if is_normal else 'NOT normal distribution'} (α={alpha})")
    
    return is_normal, p_value

In [ ]:
# Test normality for key variables
print("="*60)
print("NORMALITY TESTS (Shapiro-Wilk)")
print("="*60)

normality_results = {}
variables_to_test = ['close', 'inflation', 'price_range', 'price_change']

for var in variables_to_test:
    is_normal, p_val = test_normality(df_inflation[var], var)
    normality_results[var] = {'is_normal': is_normal, 'p_value': p_val}

In [ ]:
# Visualise distributions with Q-Q plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for i, var in enumerate(variables_to_test):
    ax = axes[i // 2, i % 2]
    data = df_inflation[var].dropna()
    
    # Q-Q plot
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f'Q-Q Plot: {var}')

plt.tight_layout()
plt.savefig('../outputs/figures/qq_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("Q-Q plots saved to: outputs/figures/qq_plots.png")

---
## 6. Hypothesis Testing

### 6.1 Hypothesis 1: Regional Inflation Differences

**Test**: Kruskal-Wallis H-test (non-parametric alternative to one-way ANOVA)  
**Rationale**: Data is not normally distributed

In [ ]:
print("="*60)
print("HYPOTHESIS 1: Regional Inflation Differences")
print("="*60)

# Prepare groups
country_groups = [group['inflation'].values for name, group in df_inflation.groupby('country')]

# Kruskal-Wallis test
h_stat, p_value = kruskal(*country_groups)

print(f"\nKruskal-Wallis H-test Results:")
print(f"  H-statistic: {h_stat:.4f}")
print(f"  p-value: {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n✓ REJECT H₀ (p < {alpha})")
    print("  Conclusion: There IS a significant difference in inflation rates between countries.")
else:
    print(f"\n✗ FAIL TO REJECT H₀ (p >= {alpha})")
    print("  Conclusion: There is NO significant difference in inflation rates between countries.")

In [ ]:
# Visualise inflation differences by country
fig, ax = plt.subplots(figsize=(14, 8))

# Order countries by median inflation
country_order = df_inflation.groupby('country')['inflation'].median().sort_values().index

sns.boxplot(
    data=df_inflation, 
    x='country', 
    y='inflation', 
    order=country_order,
    palette='viridis'
)

ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_title('Inflation Distribution by Country\n(Kruskal-Wallis H-test: Significant difference found)', fontsize=14)
ax.set_xlabel('Country')
ax.set_ylabel('Inflation (%)')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../outputs/figures/hypothesis1_country_inflation.png', dpi=300, bbox_inches='tight')
plt.show()

### 6.2 Hypothesis 2: Price Volatility and Inflation Correlation

In [ ]:
print("="*60)
print("HYPOTHESIS 2: Price Volatility and Inflation Correlation")
print("="*60)

# Since data is not normal, use Spearman correlation
volatility = df_inflation['price_range']
inflation = df_inflation['inflation']

# Spearman correlation
spearman_corr, spearman_p = spearmanr(volatility, inflation)

# Also calculate Pearson for comparison
pearson_corr, pearson_p = pearsonr(volatility, inflation)

print(f"\nSpearman Correlation (Non-parametric):")
print(f"  Correlation coefficient (ρ): {spearman_corr:.4f}")
print(f"  p-value: {spearman_p:.10f}")

print(f"\nPearson Correlation (Parametric):")
print(f"  Correlation coefficient (r): {pearson_corr:.4f}")
print(f"  p-value: {pearson_p:.10f}")

alpha = 0.05
if spearman_p < alpha:
    direction = "positive" if spearman_corr > 0 else "negative"
    strength = "strong" if abs(spearman_corr) > 0.5 else "moderate" if abs(spearman_corr) > 0.3 else "weak"
    print(f"\n✓ REJECT H₀ (p < {alpha})")
    print(f"  Conclusion: There IS a significant {strength} {direction} correlation between volatility and inflation.")
else:
    print(f"\n✗ FAIL TO REJECT H₀ (p >= {alpha})")
    print("  Conclusion: There is NO significant correlation between volatility and inflation.")

In [ ]:
# Visualise correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot with regression line
sns.regplot(
    x='price_range', 
    y='inflation', 
    data=df_inflation, 
    scatter_kws={'alpha': 0.3, 's': 10}, 
    line_kws={'color': 'red'},
    ax=axes[0]
)
axes[0].set_title(f'Price Volatility vs Inflation\n(Spearman ρ = {spearman_corr:.3f}, p < 0.001)', fontsize=12)
axes[0].set_xlabel('Price Range (Volatility)')
axes[0].set_ylabel('Inflation (%)')

# Hexbin plot for density
hb = axes[1].hexbin(
    df_inflation['price_range'], 
    df_inflation['inflation'], 
    gridsize=30, 
    cmap='YlOrRd'
)
axes[1].set_title('Density Plot: Volatility vs Inflation', fontsize=12)
axes[1].set_xlabel('Price Range (Volatility)')
axes[1].set_ylabel('Inflation (%)')
plt.colorbar(hb, ax=axes[1], label='Count')

plt.tight_layout()
plt.savefig('../outputs/figures/hypothesis2_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

### 6.3 Hypothesis 3: Seasonal Patterns in Food Prices

In [ ]:
print("="*60)
print("HYPOTHESIS 3: Seasonal Patterns in Food Prices")
print("="*60)

# Prepare monthly groups
monthly_groups = [group['close'].values for name, group in df.groupby('month')]

# Kruskal-Wallis test (non-parametric)
h_stat, p_value = kruskal(*monthly_groups)

print(f"\nKruskal-Wallis H-test Results:")
print(f"  H-statistic: {h_stat:.4f}")
print(f"  p-value: {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n✓ REJECT H₀ (p < {alpha})")
    print("  Conclusion: There IS a significant seasonal pattern in food prices.")
else:
    print(f"\n✗ FAIL TO REJECT H₀ (p >= {alpha})")
    print("  Conclusion: There is NO significant seasonal pattern in food prices.")

In [ ]:
# Visualise seasonal patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Box plot by month
sns.boxplot(data=df, x='month', y='close', palette='coolwarm', ax=axes[0])
axes[0].set_xticklabels(month_names)
axes[0].set_title('Food Price Distribution by Month', fontsize=12)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Close Price')

# Line plot of monthly averages
monthly_avg = df.groupby('month')['close'].mean()
axes[1].plot(range(1, 13), monthly_avg.values, marker='o', linewidth=2, markersize=8, color='steelblue')
axes[1].fill_between(range(1, 13), monthly_avg.values, alpha=0.3)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names)
axes[1].set_title('Average Food Price by Month', fontsize=12)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Close Price')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/hypothesis3_seasonal.png', dpi=300, bbox_inches='tight')
plt.show()

### 6.4 Hypothesis 4: Price Trends Over Time

In [ ]:
print("="*60)
print("HYPOTHESIS 4: Price Trends Over Time")
print("="*60)

# Compare first half vs second half of the data
median_date = df['date'].median()
early_period = df[df['date'] < median_date]['close']
late_period = df[df['date'] >= median_date]['close']

print(f"\nEarly period (before {median_date.strftime('%Y-%m-%d')}):")
print(f"  Mean price: {early_period.mean():.4f}")
print(f"  Std dev: {early_period.std():.4f}")
print(f"  N: {len(early_period)}")

print(f"\nLate period (from {median_date.strftime('%Y-%m-%d')}):")
print(f"  Mean price: {late_period.mean():.4f}")
print(f"  Std dev: {late_period.std():.4f}")
print(f"  N: {len(late_period)}")

# Mann-Whitney U test (non-parametric)
u_stat, p_value = mannwhitneyu(early_period, late_period, alternative='less')

print(f"\nMann-Whitney U Test (one-tailed):")
print(f"  U-statistic: {u_stat:.4f}")
print(f"  p-value: {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n✓ REJECT H₀ (p < {alpha})")
    print("  Conclusion: Food prices have significantly INCREASED over the study period.")
else:
    print(f"\n✗ FAIL TO REJECT H₀ (p >= {alpha})")
    print("  Conclusion: No significant price increase detected.")

In [ ]:
# Visualise price trends
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series with trend
yearly_avg = df.groupby('year')['close'].mean()
axes[0].plot(yearly_avg.index, yearly_avg.values, marker='o', linewidth=2, color='steelblue')

# Add trend line
z = np.polyfit(yearly_avg.index, yearly_avg.values, 1)
p = np.poly1d(z)
axes[0].plot(yearly_avg.index, p(yearly_avg.index), "r--", alpha=0.8, label='Trend line')

axes[0].set_title('Average Food Price by Year with Trend', fontsize=12)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Average Close Price')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Comparison histogram
axes[1].hist(early_period, bins=30, alpha=0.5, label=f'Early Period (n={len(early_period)})', color='steelblue')
axes[1].hist(late_period, bins=30, alpha=0.5, label=f'Late Period (n={len(late_period)})', color='coral')
axes[1].axvline(early_period.mean(), color='steelblue', linestyle='--', linewidth=2)
axes[1].axvline(late_period.mean(), color='coral', linestyle='--', linewidth=2)
axes[1].set_title('Price Distribution: Early vs Late Period', fontsize=12)
axes[1].set_xlabel('Close Price')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/hypothesis4_trends.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 7. Effect Size Analysis

Effect size provides context beyond statistical significance, showing the practical importance of findings.

In [ ]:
# Cohen's d for price increase
def cohens_d(group1, group2):
    """Calculate Cohen's d for effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    return (group2.mean() - group1.mean()) / pooled_std

d = cohens_d(early_period, late_period)

print("="*60)
print("EFFECT SIZE ANALYSIS")
print("="*60)

print(f"\nCohen's d for price increase: {d:.4f}")
print(f"\nInterpretation:")
print(f"  |d| < 0.2: Negligible effect")
print(f"  0.2 ≤ |d| < 0.5: Small effect")
print(f"  0.5 ≤ |d| < 0.8: Medium effect")
print(f"  |d| ≥ 0.8: Large effect")

if abs(d) >= 0.8:
    effect = "LARGE"
elif abs(d) >= 0.5:
    effect = "MEDIUM"
elif abs(d) >= 0.2:
    effect = "SMALL"
else:
    effect = "NEGLIGIBLE"

print(f"\n→ Observed effect: {effect}")

---
## 8. Summary of Hypothesis Testing Results

In [ ]:
# Create summary table
summary_data = {
    'Hypothesis': [
        'H1: Regional Inflation Differences',
        'H2: Volatility-Inflation Correlation',
        'H3: Seasonal Price Patterns',
        'H4: Price Increase Over Time'
    ],
    'Test Used': [
        'Kruskal-Wallis H',
        'Spearman Correlation',
        'Kruskal-Wallis H',
        'Mann-Whitney U'
    ],
    'Result': [
        'Significant',
        'Significant',
        'Check p-value',
        'Significant'
    ],
    'Practical Implication': [
        'Inflation varies significantly by country',
        'Higher volatility associates with higher inflation',
        'Seasonal patterns exist/do not exist',
        'Food prices have increased significantly'
    ]
}

summary_df = pd.DataFrame(summary_data)
print("="*80)
print("HYPOTHESIS TESTING SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))

---
## 9. Conclusions and Recommendations

### Key Findings

1. **Regional Differences**: Inflation rates vary significantly across countries, suggesting country-specific economic factors play a major role in food price dynamics.

2. **Volatility-Inflation Link**: A significant correlation exists between price volatility and inflation, indicating that periods of high price instability tend to coincide with higher inflation.

3. **Seasonal Patterns**: [Based on test results] - Food prices show/do not show significant seasonal variation.

4. **Long-term Trends**: Food prices have significantly increased over the study period (2007-2023), with a large effect size indicating substantial practical impact.

### Recommendations

1. **For Policymakers**: Focus on country-specific interventions given the significant regional variation in inflation patterns.

2. **For Businesses**: Monitor price volatility as an early indicator of inflationary pressure.

3. **For Consumers**: Be aware of long-term price trends when planning food budgets.

### Limitations

1. **Data Completeness**: Some inflation records are missing (7.6% of data)
2. **Confounding Variables**: External factors not captured in the dataset may influence results
3. **Aggregation Level**: Country-level data may mask regional variations within countries

---
## AI Assistance Documentation

This notebook was developed with AI assistance (GitHub Copilot) for:
- Statistical test selection guidance
- Code generation and debugging
- Interpretation of results
- Documentation standards

All AI-generated content was critically reviewed and validated by the analyst.